# 00 · Setup & provider check
Run this first. It installs libraries, shows which keys are configured, and (in LIVE mode) makes one tiny real call to each provider so key/model problems show up *here* instead of inside the chatbot.

**Pipeline map (notebooks are connected through the shared Chroma store):**
`01 ingest` → `02 retrieve` → `03 augment + generate` → `04 agentic LangGraph` → `05 evaluate`

In [ ]:
# Install every library the pipeline needs (safe to re-run)
%pip install -q -r ../requirements.txt

In [ ]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
from dotenv import load_dotenv
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# Automatically load environment variables from rag_service/.env and repo-root .env
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT.parent / ".env", override=False)

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env (default when keys are present)
has_keys = bool(os.getenv("GEMINI_API_KEY") or os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY"))
OFFLINE = (os.getenv("NB_OFFLINE", "0" if has_keys else "1") == "1")
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings, reset_settings_cache
reset_settings_cache()
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

## Which keys are present? (values are never printed)

In [ ]:
for k in ["GEMINI_API_KEY","GROQ_API_KEY","OPENAI_API_KEY","CHROMA_API_KEY","CHROMA_TENANT","CHROMA_DATABASE","INTERNAL_SERVICE_TOKEN"]:
    print(f"{k:24s}", "SET" if os.getenv(k) else "-- missing --")

## Embedding model smoke test

In [ ]:
from app.providers import get_embeddings, embedding_id
emb = get_embeddings()
v = emb.embed_query("What is photosynthesis?")
print("embedding id:", embedding_id(), "| dimension:", len(v))
assert len(v) > 100

## LLM smoke test (light + heavy tier)

In [ ]:
from app.providers import get_llm, llm_text
for tier in ("light", "heavy"):
    out = llm_text(get_llm(tier).invoke("Reply with the single word: pong"))
    print(tier, "->", out.strip()[:60])

## Vector store connectivity (write → read → delete a probe record)

In [ ]:
from app.store import get_store
from langchain_core.documents import Document
st = get_store()
print("mode:", st.mode, "| collection:", st.collection_name, "| heartbeat:", st.heartbeat())
probe = Document(page_content="probe record for connectivity test", metadata={"chunk_id":"__probe__:0","document_id":"__probe__","chunk_index":0,"subject_code":"__PROBE__","is_active":True})
st.upsert([probe]); assert any(d.metadata["document_id"]=="__probe__" for d in st.chunks_of("__probe__"))
st.delete_document("__probe__"); print("probe write/read/delete OK; total chunks now:", st.count())